In [ ]:
import random
import json
from datetime import datetime
import pycountry
import pandas as pd
import seaborn as sns

from emu_renewal.constants import DATA_PATH, FULL_RUN, TIMEOUTS, RERUNS, OUTPUTS_PATH
from emu_renewal.run import run_single_country
from emu_renewal.outputs import get_param_vals_by_analysis
from emu_renewal.utils import get_analysis_paths

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, countries)
n_samples = 6
sample_countries = random.sample(list(analysis_paths), n_samples)
sample_paths = {c: {} for c in sample_countries}
for iso3 in sample_countries:
    sample_mob_type = random.sample(list(analysis_paths[iso3]), 1)[0]
    sample_paths[iso3][sample_mob_type] = analysis_paths[iso3][sample_mob_type]

In [ ]:
for iso3 in sample_paths:
    mob_type = list(sample_paths[iso3].keys())[0]
    run_single_country(iso3, mob_type, "suscept", False, prog_bar=True)

In [ ]:
from matplotlib import pyplot as plt
import pycountry

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
flat_axes = axes.ravel()
param = "dispersion_proc"
for c, iso3 in enumerate(sample_paths):
    mob_type = list(sample_paths[iso3].keys())[0]
    run_paths = {
        "waning": {mob_type: OUTPUTS_PATH / "suscept" / iso3 / mob_type},
        "no_waning": analysis_paths[iso3],
    }    
    posts = [get_param_vals_by_analysis(param, p)[mob_type] for p in run_paths.values()]
    combined_disps = pd.concat(posts, axis=1)
    combined_disps.columns = run_paths.keys()
    ax = flat_axes[c]
    sns.kdeplot(combined_disps, ax=ax, fill=True, alpha=0.1, linewidth=1.5, common_norm=False)
    ax.set_title(pycountry.countries.lookup(iso3).name)
    ax.set_yticks([])
    ax.set_ylabel("")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
flat_axes = axes.ravel()
for c, iso3 in enumerate(sample_paths):
    mob_type = list(sample_paths[iso3].keys())[0]
    run_paths = {
        "waning": {mob_type: OUTPUTS_PATH / "suscept" / iso3 / mob_type},
        "no_waning": analysis_paths[iso3],
    }
    procs = {p: pd.read_hdf(run_paths[p][mob_type] / "spaghetti.h5")["process"] for p in run_paths}
    quants = {p: procs[p].quantile([0.5], axis=1).T for p in run_paths}
    
    ax = flat_axes[c]
    for p in run_paths:
        ax.plot(quants[p].index, quants[p], label=p)
    ax.legend()
    ax.set_title(pycountry.countries.lookup(iso3).name)
    ax.set_yticks([])
    ax.set_ylabel("")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
fig.tight_layout()